# Notebook 07 of 7 — Offline Recording + End-to-End

*Portfolio Intelligence Engine — User Guide Series.*
[Series README](README.md) · [Story Bible](STORY_BIBLE.md) · Filed under
epic [#1352](https://github.com/prajoria/OpenBB/issues/1352).

---

## Where we are in Sam's story

This is the capstone. Six notebooks ago I met the plumbing. Now I want the whole thing to work when the internet doesn't, when a provider is down, and six months from today. This is where `scrape_record` becomes the whole point of the fork — not a workaround, a design.

By the end of this notebook we will be able to answer one question:

> *Can I rebuild my Monday-morning analysis in 45 minutes on a plane from a git checkout, and can I do it exactly the same way in six months?*


In [1]:
# [Phase B / NB07 §0] environment sanity — assert .venv_portfolio + STATE dir
import sys, pathlib
assert "venv_portfolio" in sys.executable, (
    "Portfolio notebooks require .venv_portfolio. See NB01 §0 for setup."
)
STATE = pathlib.Path(".notebook_state")
STATE.mkdir(exist_ok=True)
print(f"Python:                   {sys.version.split()[0]}")
print(f"Interpreter:              {pathlib.Path(sys.executable).name}")
print(f"venv sanity check:        passed (interpreter path contains 'venv_portfolio')")
print(f"State dir (repo-rel):     {STATE}/")


Python:                   3.12.10
Interpreter:              python.exe
venv sanity check:        passed (interpreter path contains 'venv_portfolio')
State dir (repo-rel):     .notebook_state/


## 1. Why offline

Three failure modes I want to eliminate:

- **Provider outage.** FMP has bad days. Yahoo has bad days. When
  they do, my process should not stop.
- **Rate limits.** Backtests that call live endpoints hit rate limits
  at the exact moment you want to iterate.
- **Reproducibility.** Six months from now when I want to know why I
  placed a trade, "MSFT quote on 2026-07-22" needs to still return the
  same number.

The answer to all three is **snapshots on disk**, checked into git,
served by fetchers that never touch the wire at query time. That is
what `scrape_record` provides for the endpoints without free live
JSON, and what `fmp_cached`'s cassette layer provides for the FMP
endpoints.

## 2. `scrape_record` — the framework

Two commands you'll actually use:

- **`scrape-record record <name> --symbol <SYM>`** — Playwright drives
  the target site (headed or headless), captures the response, writes
  a JSON snapshot to `openbb_platform/tools/scrape_record/snapshots/`.
- **`scrape-record replay <name> --symbol <SYM>`** — reads the JSON,
  passes it through the extractor, returns the same shape the live
  path would have. This is what fetchers call under the hood.

Two auxiliary commands:

- **`scrape-record list`** — every recording available.
- **`scrape-record verify`** — sanity-check every checked-in snapshot
  against its extractor (catches schema drift after a fetcher change).

*The code cell below runs `scrape-record list` and `verify` against
every snapshot in the repo.*

For batch refresh across the whole notebook-series universe, use the
sweep script: `python scripts/record_universe_snapshots.py`. It reads
`notebooks/portfolio/UNIVERSE.md`, plans (endpoint × ticker) captures
per ticker classification (equity ETFs get holdings; bond ETFs get
bond-holdings; commodities skip holdings entirely), and skips
snapshots whose envelope `captured_at` is fresher than 7 days.

In [2]:
# [Phase B / NB07 §2] scrape-record CLI probe — list + spot-verify
# Shell out to the CLI; DO NOT hang the notebook if it's missing.
import subprocess, sys, shutil
from collections import Counter
from pathlib import Path

def _cli() -> list[str]:
    """Resolve scrape-record CLI, preferring the venv shim."""
    exe_dir = Path(sys.executable).parent
    for name in ("scrape-record.exe", "scrape-record"):
        candidate = exe_dir / name
        if candidate.exists():
            return [str(candidate)]
    found = shutil.which("scrape-record")
    if found:
        return [found]
    return [sys.executable, "-c", "from scrape_record.cli import main; main()"]

cli = _cli()
try:
    proc = subprocess.run(cli + ["list"], capture_output=True, text=True, timeout=30)
except (FileNotFoundError, subprocess.TimeoutExpired) as exc:
    print(f"WARNING: scrape-record CLI unavailable ({type(exc).__name__}: {exc}).")
    print("  Skipping snapshot probe. Install with: pip install -e openbb_platform/tools/scrape_record")
else:
    if proc.returncode != 0:
        print(f"WARNING: `scrape-record list` exited {proc.returncode}")
        print((proc.stderr or proc.stdout)[:400])
    else:
        lines = [ln for ln in proc.stdout.splitlines() if ln.strip()]
        # first line is header
        counts: Counter[str] = Counter()
        total = 0
        for ln in lines[1:]:
            parts = ln.split()
            if len(parts) >= 3:
                counts[parts[0]] += 1
                total += 1
        print(f"Snapshots on disk: {total} across {len(counts)} recording types")
        print()
        print(f"  {'Recording':<40} {'Count':>6}")
        print(f"  {'-'*40} {'-'*6}")
        for name, n in counts.most_common():
            print(f"  {name:<40} {n:>6}")

        # Spot-verify one snapshot per recording type to catch extractor drift.
        print()
        print("Spot-verify (one snapshot per recording type):")
        drift = 0
        for rec_name in counts:
            # find a symbol from the earlier list output for this recording
            symbol = None
            for ln in lines[1:]:
                parts = ln.split()
                if parts and parts[0] == rec_name:
                    symbol = parts[1]
                    break
            if not symbol:
                continue
            try:
                vp = subprocess.run(
                    cli + ["verify", rec_name, "--symbol", symbol],
                    capture_output=True, text=True, timeout=30,
                )
                ok = vp.returncode == 0
                marker = "ok" if ok else "DRIFT"
                if not ok:
                    drift += 1
                print(f"  {rec_name:<40} {symbol:<8}  {marker}")
                if not ok:
                    tail = (vp.stderr or vp.stdout).strip().splitlines()[-1:]
                    for t in tail:
                        print(f"      → {t[:120]}")
            except subprocess.TimeoutExpired:
                print(f"  {rec_name:<40} {symbol:<8}  TIMEOUT")
                drift += 1
        print()
        if drift == 0:
            print("No drift detected — all recording extractors round-trip cleanly.")
        else:
            print(f"WARNING: {drift} recording(s) failed spot-verify.")


Snapshots on disk: 120 across 5 recording types

  Recording                                 Count
  ---------------------------------------- ------
  yahoo_equity_info                            55
  yahoo_equity_quote                           55
  yahoo_etf_holdings                            8
  yahoo_bond_etf_holdings                       1
  yahoo_options_chain                           1

Spot-verify (one snapshot per recording type):


  yahoo_bond_etf_holdings                  BND       ok


  yahoo_equity_info                        AAPL      ok


  yahoo_equity_quote                       AAPL      ok


  yahoo_etf_holdings                       DIA       ok


  yahoo_options_chain                      AAPL      ok

No drift detected — all recording extractors round-trip cleanly.


## 3. Path-traversal defense — a security aside

Symbol strings reach file I/O (`snapshots/<name>/<SYMBOL>.json`).
That's a path-traversal vector if you're not careful. PR #1351 landed
defense-in-depth:

1. **Pydantic pattern** on every fetcher's `symbol` field —
   `^[A-Za-z0-9._\-^=]{1,32}$` — catches slashes and NUL at the API
   layer.
2. **Config-level allowlist** in `scrape_record.config._validate_snapshot_component`
   — rejects `.`, `..`, NUL, and anything outside a 64-char allowlist.
3. **`.resolve()` + `.relative_to()` assertion** in
   `snapshot_path()` — even if the first two are bypassed, the
   resolved path must live inside `snapshots_dir` or the call raises.

You will not run into this in normal use. But it's the shape of
security aside that gets a fork through a review, and it's why the
symbol allowlist looks over-strict.

## 4. `portfolio_export` — the sibling tool

`scrape_record` records **public** data into the repo. `portfolio_export`
is its sibling for **private** data — brokerage exports (Fidelity
positions, Schwab tax lots, and so on). Different rules:

- **`scrape_record` snapshots** live **inside** the repo (public data,
  reproducible builds).
- **`portfolio_export` downloads** live **outside** the repo, at
  `H:\masterswork\browser_exports\` on this machine (private data,
  never committed).

The `portfolio_export` CLI has parallel commands (`record`, `replay`,
`csvs`, `inspect`, `list`, `status`, `tag`) but everything it produces
is gitignored by design. Sam won't run it in this series — it's for the
techtrade lane. But it's why the config-layer distinction exists (see
`portfolio_export/config.py:_validate_outside_repo` vs
`scrape_record/config.py:_validate_snapshots_inside_package`).

## 5. The Monday-morning routine — end to end

Now the capstone. One flow that reloads state from NB01-NB06 and
produces one HTML report. This is what Sam actually runs on a Monday
morning:

1. Confirm environment (from NB01)
2. Deep-dive the most-changed name from Friday close (NB02 pipeline)
3. X-ray + risk on the current basket (NB03)
4. 30-day events + fresh smart-money rollup (NB04)
5. What-if the trade candidates that fall out; paper-trade the ones
   that survive (NB05)
6. Backtest any strategy about to move real capital (NB06)
7. Record any snapshots that got stale (NB07 §2)

Elapsed: about 45 minutes if nothing surprising surfaces. Longer if
NB03 flags concentration that NB05 wasn't planning to address.

*The code cell below runs the whole routine, using the artifacts from
`.notebook_state/`, and writes `portfolio_report.html`.*

In [3]:
# [Phase B / NB07 §5] End-to-end capstone — reload artifacts + compose HTML report
# Reads every NB01-NB06 pickle/json from .notebook_state/ and writes a single
# self-contained HTML file. Missing artifacts trigger a WARNING and section
# skip — never a crash. Pickle safety: trusted-local-only per repo policy.
import json
import pickle  # noqa: S403  # trusted local; .notebook_state/ is gitignored
import html
import sys
from pathlib import Path
sys.path.insert(0, "../../Analysis")  # so Phase7Result unpickles cleanly

STATE = Path(".notebook_state")
OUT = STATE / "portfolio_report.html"

def _load_pickle(name: str):
    p = STATE / name
    if not p.exists():
        print(f"WARNING: {name} missing — skipping section")
        return None
    try:
        return pickle.loads(p.read_bytes())  # noqa: S301
    except Exception as exc:
        print(f"WARNING: {name} unpickle failed ({exc}) — skipping section")
        return None

def _load_json(name: str):
    p = STATE / name
    if not p.exists():
        print(f"WARNING: {name} missing — skipping section")
        return None
    try:
        return json.loads(p.read_text(encoding="utf-8"))
    except Exception as exc:
        print(f"WARNING: {name} parse failed ({exc}) — skipping section")
        return None

basket = _load_json("basket.json")
msft_p7 = _load_pickle("msft_p7.pkl")
xray = _load_pickle("xray.pkl")
smart_money = _load_pickle("smart_money.pkl")
paper = _load_pickle("paper_blotter.pkl")
backtest = _load_pickle("backtest_result.pkl")

def esc(v) -> str:
    return html.escape(str(v))

CSS = """
body { font-family: -apple-system, Segoe UI, Roboto, sans-serif; max-width: 960px;
       margin: 2em auto; padding: 0 1em; color: #222; line-height: 1.5; }
h1 { border-bottom: 3px solid #2b6cb0; padding-bottom: .3em; }
h2 { color: #2b6cb0; margin-top: 2em; border-bottom: 1px solid #cbd5e0;
     padding-bottom: .2em; }
.skip { color: #a0522d; font-style: italic; }
table { border-collapse: collapse; margin: 1em 0; }
th, td { border: 1px solid #cbd5e0; padding: .35em .7em; text-align: left; }
th { background: #edf2f7; }
.metric { display: inline-block; margin-right: 1.5em; }
.metric .k { color: #4a5568; font-size: .85em; }
.metric .v { font-weight: bold; font-size: 1.1em; }
.warn { color: #b7410e; }
.footer { color: #718096; font-size: .85em; margin-top: 3em;
          border-top: 1px solid #cbd5e0; padding-top: 1em; }
"""

parts: list[str] = []
parts.append("<!doctype html><html><head><meta charset='utf-8'>")
parts.append("<title>Portfolio Intelligence Report</title>")
parts.append(f"<style>{CSS}</style></head><body>")
parts.append("<h1>Portfolio Intelligence — Monday-Morning Report</h1>")
parts.append("<p><em>Composed from NB01–NB06 artifacts in <code>.notebook_state/</code>. "
             "This is the reunion artifact — one page, six sections, one flow.</em></p>")

# --- 1. Basket ---
parts.append("<h2>1. Basket (NB01/NB03)</h2>")
if basket:
    rows = sorted(basket, key=lambda r: -r.get("weight", 0))[:12]
    parts.append("<table><tr><th>Symbol</th><th>Weight</th><th>Kind</th><th>Note</th></tr>")
    for r in rows:
        sym = r.get("symbol") or r.get("ticker", "?")
        parts.append(
            f"<tr><td>{esc(sym)}</td><td>{r.get('weight', 0)*100:.1f}%</td>"
            f"<td>{esc(r.get('kind',''))}</td><td>{esc(r.get('note',''))}</td></tr>"
        )
    parts.append("</table>")
else:
    parts.append("<p class='skip'>basket.json missing — section skipped.</p>")

# --- 2. Deep dive ---
parts.append("<h2>2. MSFT Deep Dive (NB02)</h2>")
if msft_p7 is not None:
    action = getattr(msft_p7, "action_label", "?")
    comp = getattr(msft_p7, "composite_score", "?")
    entry = getattr(msft_p7, "entry_quality", "?")
    parts.append(
        f"<p><span class='metric'><span class='k'>Action</span><br>"
        f"<span class='v'>{esc(action)}</span></span>"
        f"<span class='metric'><span class='k'>Composite score</span><br>"
        f"<span class='v'>{esc(f'{comp:.3f}' if isinstance(comp,(int,float)) else comp)}</span></span>"
        f"<span class='metric'><span class='k'>Entry quality</span><br>"
        f"<span class='v'>{esc(entry)}</span></span></p>"
    )
    override = getattr(msft_p7, "hard_override", None)
    if override:
        parts.append(f"<p class='warn'>Hard override:{esc(override)}</p>")
else:
    parts.append("<p class='skip'>msft_p7.pkl missing — section skipped.</p>")

# --- 3. X-Ray ---
parts.append("<h2>3. Basket X-Ray (NB03)</h2>")
if xray:
    eff = xray.get("effective_positions", {})
    sectors = xray.get("xray_by_sector", {}) or xray.get("naive_by_sector", {})
    hhi = xray.get("hhi_xray") or xray.get("hhi_raw")
    neff = xray.get("neff_xray") or xray.get("neff_raw")
    top_sectors = sorted(sectors.items(), key=lambda kv: -kv[1])[:3]
    parts.append(f"<p><span class='metric'><span class='k'>Effective positions</span><br>"
                 f"<span class='v'>{len(eff)}</span></span>"
                 f"<span class='metric'><span class='k'>HHI</span><br>"
                 f"<span class='v'>{hhi:.4f}</span></span>"
                 f"<span class='metric'><span class='k'>Effective-N</span><br>"
                 f"<span class='v'>{neff:.2f}</span></span></p>")
    parts.append("<p><strong>Top-3 sectors:</strong></p><table><tr><th>Sector</th><th>Weight</th></tr>")
    for name, w in top_sectors:
        parts.append(f"<tr><td>{esc(name)}</td><td>{w*100:.2f}%</td></tr>")
    parts.append("</table>")
else:
    parts.append("<p class='skip'>xray.pkl missing — section skipped.</p>")

# --- 4. Events & Smart Money ---
parts.append("<h2>4. Events & Smart Money (NB04)</h2>")
if smart_money:
    top = smart_money.get("top_conviction", []) or []
    warns = smart_money.get("warnings", []) or []
    if top:
        parts.append("<table><tr><th>Symbol</th><th>Composite</th><th>Signals</th></tr>")
        for it in top[:5]:
            sym = getattr(it, "symbol", "?")
            comp = getattr(it, "composite", 0.0)
            n = getattr(it, "signal_count", 0)
            parts.append(f"<tr><td>{esc(sym)}</td><td>{comp:+.2f}</td><td>{n}</td></tr>")
        parts.append("</table>")
    else:
        parts.append("<p><em>No smart-money conviction (upstream sources returned empty on current sub-tier).</em></p>")
    if warns:
        parts.append("<p class='warn'><strong>Warnings:</strong></p><ul>")
        for w in warns:
            parts.append(f"<li>{esc(w)}</li>")
        parts.append("</ul>")
else:
    parts.append("<p class='skip'>smart_money.pkl missing — section skipped.</p>")

# --- 5. What-If & Paper ---
parts.append("<h2>5. What-If & Paper Blotter (NB05)</h2>")
if paper:
    trades = paper.get("trades_submitted", []) or []
    wf = paper.get("whatif_diff", {}) or {}
    end_cash = paper.get("ending_cash")
    start_cash = paper.get("starting_cash")
    parts.append(f"<p><span class='metric'><span class='k'>Trades submitted</span><br>"
                 f"<span class='v'>{len(trades)}</span></span>"
                 f"<span class='metric'><span class='k'>HHI Δ</span><br>"
                 f"<span class='v'>{wf.get('hhi_delta', 0):+.4f}</span></span>"
                 f"<span class='metric'><span class='k'>Ending cash</span><br>"
                 f"<span class='v'>${end_cash:,.2f}</span></span>"
                 f"<span class='metric'><span class='k'>Starting cash</span><br>"
                 f"<span class='v'>${start_cash:,.2f}</span></span></p>")
    if trades:
        parts.append("<table><tr><th>Symbol</th><th>Action</th><th>ΔShares</th><th>Rationale</th></tr>")
        for t in trades:
            parts.append(
                f"<tr><td>{esc(t.get('symbol','?'))}</td>"
                f"<td>{esc(t.get('action',''))}</td>"
                f"<td>{esc(t.get('delta_shares',''))}</td>"
                f"<td>{esc(t.get('rationale',''))}</td></tr>"
            )
        parts.append("</table>")
else:
    parts.append("<p class='skip'>paper_blotter.pkl missing — section skipped.</p>")

# --- 6. Backtest & Validation ---
parts.append("<h2>6. Backtest & Validation (NB06)</h2>")
if backtest:
    m = backtest.get("metrics", {}) or {}
    sweep = backtest.get("sweep_best", {}) or {}
    val = backtest.get("validation", {}) or {}
    pbo = val.get("pbo")
    if pbo is None:
        verdict = "n/a"
    elif pbo < 0.3:
        verdict = "reasonable edge (PBO < 0.3)"
    elif pbo < 0.5:
        verdict = "borderline (0.3 ≤ PBO < 0.5)"
    elif pbo < 0.7:
        verdict = "coin flip (0.5 ≤ PBO < 0.7)"
    else:
        verdict = "likely overfit (PBO ≥ 0.7)"
    parts.append(f"<p><span class='metric'><span class='k'>Sharpe</span><br>"
                 f"<span class='v'>{m.get('sharpe', float('nan')):.3f}</span></span>"
                 f"<span class='metric'><span class='k'>MaxDD</span><br>"
                 f"<span class='v'>{m.get('max_drawdown', float('nan')):.2%}</span></span>"
                 f"<span class='metric'><span class='k'>CAGR</span><br>"
                 f"<span class='v'>{m.get('cagr', float('nan')):.2%}</span></span></p>")
    parts.append(f"<p><strong>Sweep best:</strong> Sharpe {sweep.get('sharpe', float('nan')):.3f} "
                 f"at params {esc(sweep.get('params', {}))}</p>")
    parts.append(f"<p><strong>PBO:</strong> {esc(f'{pbo:.4f}' if isinstance(pbo,(int,float)) else pbo)} "
                 f"— <em>{esc(verdict)}</em></p>")
else:
    parts.append("<p class='skip'>backtest_result.pkl missing — section skipped.</p>")

parts.append("<div class='footer'>Generated by NB07 (portfolio-intel series). "
             "Repo-relative paths only; no external CDNs; safe to view offline.</div>")
parts.append("</body></html>")

html_doc = "".join(parts)
OUT.write_text(html_doc, encoding="utf-8")
size = OUT.stat().st_size
print(f"Wrote (repo-rel): .notebook_state/portfolio_report.html  ({size:,} bytes)")


Wrote (repo-rel): .notebook_state/portfolio_report.html  (4,938 bytes)


## 6. The widget map

The whole engine ships a widget frontend
(`openbb_platform/extensions/portfolio_intel/openbb_portfolio_intel/widget_backend/`)
that the desktop app renders. Every widget is served by one or two
routers you now know:

| Widget (execution-plan milestone) | Router | Introduced in notebook |
|---|---|---|
| Sector look-through pie (M2) | `xray_router.look_through` | NB03 §4 |
| Country look-through pie (M2) | `xray_router.look_through` | NB03 §4 |
| Event calendar timeline (M2) | `events_router` | NB04 §3 |
| Smart-money ribbon (M2) | `smart_money_router.rollup` | NB04 §4 |
| Risk dashboard (M2) | `risk_router.metrics` + `.concentration` | NB03 §6-7 |
| What-if diff (M3) | `portfolio_intel_router` (what-if) | NB05 §3 |
| Brinson attribution (M3) | `portfolio_intel_router` (attribution) | NB05 §5 |
| Paper blotter (M3) | `paper_alerts_router` | NB05 §6 |
| Paper positions (M3) | `paper_alerts_router` | NB05 §6 |
| Paper account summary (M3) | `paper_alerts_router` | NB05 §6 |
| Alerts panel (M4) | `paper_alerts_router.alerts` | NB05 §7 |
| News sentiment (M4) | `news_sentiment_router` | NB04 §6 |
| Backtest-this-portfolio (M4) | `backtest_router` | NB06 §2 |
| Tearsheet (M4) | `backtest.tearsheet` | NB06 §6 |

If you're wondering "where does the code that renders widget X live?"
— it's the router in the middle column. Every widget in the frontend
is one you've called directly from Python in this series.

## 7. Where to go next

The natural directions after this series:

- **`widget_backend/`** — how the routers above serve the frontend.
  The React side lives in `desktop/`.
- **`openbb_platform/extensions/mcp_server/`** — the platform's MCP
  server. Driving it from Claude/Cursor/etc. is a separate guide.
- **Upstream promotion policy.** The `portfolio` branch periodically
  absorbs `develop`; upstream promotion (portfolio → develop) is
  user-initiated only. `CLAUDE.md` at the fork root has the rules.
- **The techtrade lane.** `notebooks/01-06` covers the techtrade
  pipeline — signals, plans, execution — for the same trader from a
  different angle.

## 8. What Sam thinks now (closing)

Six weeks in, running this routine every Monday morning:

- The 10-ticker basket is now 8 tickers. Sam closed AMD and rotated
  half of QQQ into VTV (value tilt) after NB03 kept flagging Tech
  concentration.
- Two paper strategies have been running long enough to have a real
  OOS Sharpe. One has PBO 0.38 and Sam is considering going live at
  1/4 size. The other has PBO 0.71 and got dropped.
- The Sunday sheet is gone. The Monday routine replaced it. It's not
  faster — it's the same 45 minutes — but the routine tells Sam
  things the sheet never did.
- The single largest behavior change: Sam no longer opens positions
  without the NB02 pipeline producing at least "Fair" entry quality.
  That alone cut Sam's turnover by more than half and the
  underperformance vs. SPY closed to about 3 points.

---

## What is NOT in this notebook

- **The MCP client-side walkthrough.** A follow-on guide.
- **The widget-authoring walkthrough** (React side). Lives in `desktop/`, not here.
- **Live-broker adapter authoring.** The paper engine has the shape a real adapter would plug into; the guide for that is future work.

## Preview of the series wrap

You have the whole engine now. There is no NB08 — this is the reunion chapter. When you want to add a widget, extend a router, record a new provider snapshot, or file a bug — start from the corresponding notebook here and follow the router path outward. Every line of the surface has a home in one of these seven files.
